In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [4]:
train = pd.read_csv('../data/labels/train_sent_emo.csv')
manifest = pd.read_csv('../data/manifest.csv')

In [3]:
X_audio = np.load("../cache/audio_wavlm_13L.npy").astype(np.float32)
tr = (manifest.split == "train").values
va = (manifest.split == "dev").values
te = (manifest.split == "test").values

X_train = torch.from_numpy(X_audio[tr])
X_val   = torch.from_numpy(X_audio[va])
X_test  = torch.from_numpy(X_audio[te])

In [4]:
EMOTIONS = {"neutral":0, "joy":1, "surprise":2, "anger":3, "sadness":4, "disgust":5, "fear":6}
manifest['emo_encoded'] = manifest["Emotion"].map(EMOTIONS)

In [5]:
Y_train = manifest[manifest.split == 'train'].emo_encoded.to_numpy()
Y_train = torch.from_numpy(Y_train)

Y_val = manifest[manifest.split == 'dev'].emo_encoded.to_numpy()
Y_val = torch.from_numpy(Y_val)

Y_test = manifest[manifest.split == 'test'].emo_encoded.to_numpy()
Y_test = torch.from_numpy(Y_test)

In [6]:
X_train.size(), Y_train.size()

(torch.Size([9988, 19968]), torch.Size([9988]))

In [7]:
class EmotionClassifier(nn.Module):

    def __init__(self):
        super().__init__()


        self.network = nn.Sequential(
            nn.Linear(X_train.size()[1], 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(.5),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(.5),

            nn.Linear(128, len(EMOTIONS))
        )

    def forward(self, x):
        return self.network(x)


counts = torch.bincount(Y_train)
class_weights = (1.0 / counts.float().sqrt())
class_weights = class_weights / class_weights.sum() * len(EMOTIONS)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)




In [8]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score


model = EmotionClassifier()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

train_ds = TensorDataset(X_train, Y_train)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)

best_val_loss = -1
patience = 50
patience_counter = 0
best_weights = {k: v.clone() for k, v in model.state_dict().items()}


best_val_loss, best_weights, stale = float("inf"), None, 0
patience = 10

for epoch in range(100):
    model.train()
    for xb, yb in train_dl:
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(X_val), Y_val).item()
        val_f1 = f1_score(Y_val, model(X_val).argmax(1), average="weighted")

    if val_loss < best_val_loss:
        best_val_loss, stale = val_loss, 0
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        flag = " *"
    else:
        stale += 1
        flag = ""
    print(f"epoch {epoch:3d}  val_loss {val_loss:.4f}  dev_wF1 {val_f1:.4f}{flag}")
    if stale >= patience:
        break

model.load_state_dict(best_weights)

epoch   0  val_loss 1.8528  dev_wF1 0.1980 *
epoch   1  val_loss 1.8033  dev_wF1 0.3294 *
epoch   2  val_loss 1.7162  dev_wF1 0.4377 *
epoch   3  val_loss 1.7048  dev_wF1 0.4153 *
epoch   4  val_loss 1.6890  dev_wF1 0.4066 *
epoch   5  val_loss 1.7260  dev_wF1 0.4039
epoch   6  val_loss 1.6642  dev_wF1 0.4451 *
epoch   7  val_loss 1.6620  dev_wF1 0.4214 *
epoch   8  val_loss 1.6712  dev_wF1 0.4435
epoch   9  val_loss 1.7443  dev_wF1 0.3715
epoch  10  val_loss 1.6138  dev_wF1 0.4451 *
epoch  11  val_loss 1.6460  dev_wF1 0.4388
epoch  12  val_loss 1.6369  dev_wF1 0.4190
epoch  13  val_loss 1.6421  dev_wF1 0.4322
epoch  14  val_loss 1.6398  dev_wF1 0.4449
epoch  15  val_loss 1.6379  dev_wF1 0.4538
epoch  16  val_loss 1.6325  dev_wF1 0.4596
epoch  17  val_loss 1.6763  dev_wF1 0.4436
epoch  18  val_loss 1.6678  dev_wF1 0.3796
epoch  19  val_loss 1.6555  dev_wF1 0.4438
epoch  20  val_loss 1.6153  dev_wF1 0.4447


<All keys matched successfully>

In [9]:

model.eval()
with torch.no_grad():
    tr_pred = model(X_train).argmax(1)
    va_pred = model(X_val).argmax(1)

print("train wF1", f1_score(Y_train, tr_pred, average="weighted"))
print("dev   wF1", f1_score(Y_val, va_pred, average="weighted"))

train wF1 0.5334054422175565
dev   wF1 0.4451084704415348


In [10]:
from sklearn.metrics import classification_report
print(classification_report(Y_val, va_pred, target_names=EMOTIONS, zero_division=0))

              precision    recall  f1-score   support

     neutral       0.58      0.68      0.63       469
         joy       0.28      0.09      0.13       163
    surprise       0.36      0.48      0.41       150
       anger       0.47      0.53      0.50       153
     sadness       0.33      0.40      0.36       111
     disgust       0.00      0.00      0.00        22
        fear       0.00      0.00      0.00        40

    accuracy                           0.48      1108
   macro avg       0.29      0.31      0.29      1108
weighted avg       0.43      0.48      0.45      1108



In [13]:
torch.save(model.state_dict(), "../cache/audio_model.pt")

In [5]:
i = 1003
row = manifest.iloc[i]
print(row.split, row.audio_path, row.Utterance, row.Emotion)

train data/audio/train/dia109_utt7.flac I mean you’d think he’d be worried about me not showing up at his club. sadness
